# Generating DEM Tiles from Classified LiDAR Data

This notebook demonstrates how to process a large collection of classified LiDAR files (`.las` / `.laz`) into individual **Digital Elevation Model (DEM) tiles** using [PDAL](https://pdal.io).

### Workflow:

1. **File search**: Recursively search for all `.las` and `.laz` files in the working directory or sub-folders.
2. **DEM parameters**: Define interpolation settings such as resolution, output type (IDW), search radius, power parameter, and window size.
3. **Per-file processing**: For each LiDAR file:

   * Read only **ground-classified points** (classification code = 2).
   * Apply **statistical outlier removal** to clean the point cloud.
   * Interpolate to raster with **Inverse Distance Weighting (IDW)**, creating one DEM tile per input file.
4. **Output**: Save all DEM tiles into a dedicated subdirectory (`data/dem_tiles`).

Tile-based approach was chosen because it is well-suited for handling very large LiDAR datasets (which we have), since each file is processed independently without overloading RAM. The resulting DEM tiles will later be mosaicked or divided into smaller patches for machine learning workflow.


## Environment Setup and Imports

On Windows I recommend to use a dedicated **Conda environment** for this workflow, because installing **PDAL** and its dependencies can be problematic on Windows.
With Conda, installation is much simpler since most geospatial libraries (PDAL, GDAL, etc.) are available via the `conda-forge` channel.

Example environment creation:

```bash
conda create -n lidar-env python=3.11 -c conda-forge pdal numpy jupyter pathlib json
conda activate lidar-env
```

Once the environment is active, you can import the necessary Python libraries in the notebook:

In [23]:
import pdal
from pathlib import Path
import json

File search

In [24]:
# Current working directory absolute path
data_dir = Path().resolve()
# Recursive search of *.laz or *.las files in all sub-folders
las_files = list(data_dir.rglob("*.laz")) + list(data_dir.rglob("*.las"))
las_files = [str(f) for f in las_files]
print(f"Found {len(las_files)} files")

Found 3 files


Define DEM parameters

In [43]:
# DEM parameters
resolution = 0.5
output_type = "min" # or "idw", but "min" probably better for our purpose
radius = 1.0 # 1 m, we can try different values
power = 2.0 # only for idw
window_size = 5 # if there are no points in radius, how many surrounding values use for interpolation

Set output directory

In [26]:
# set output dir
output_dir = data_dir / "data" / "dem_tiles"
# create data folder if it doesn't exist
output_dir.mkdir(exist_ok=True)

Process each file using PDAL pipeline and save as DEM tiles

In [45]:
# For each file (enumerate just so we can track how many files has been processed)
for i, las in enumerate(las_files, 1):
    las_path = Path(las)
    # name of DEM file
    # dem_file = output_dir / f"{las_path.stem}_dem.tif"
    dem_file = output_dir / f"{las_path.stem}_dem_{output_type}.tif"
    # Tracking progress
    print(f"[{i}/{len(las_files)}] Processing {las_path.name} → {dem_file.name}")

    # input for PDAL is a JSON which can be done as a dictionary in python and then converting to JSON
    pipeline_dict = {
        "pipeline": [{"type": "readers.las", "filename": str(las_path)}, # read file     
            {"type": "filters.range", "limits": "Classification[2:2]"},  # only ground class
            {"type": "filters.outlier", "method": "statistical", "mean_k": 8, "multiplier": 2.5}, # filter outlying points
            {
                "type": "writers.gdal",  # create DEM with chosen parameters
                "filename": str(dem_file),
                "resolution": resolution,
                "output_type": output_type,
                "radius": radius,
                "power": power,
                "window_size": window_size,
                "gdaldriver": "GTiff"
            }
        ]
    }
    # convert dictionary to JSON and run pipeline
    pipeline = pdal.Pipeline(json.dumps(pipeline_dict))
    count = pipeline.execute()

print(f"Pipeline finished. Created {len(las_files)} DEM files")

[1/3] Processing P4431G1_1.laz → P4431G1_1_dem_min.tif
[2/3] Processing P4431G1_2.laz → P4431G1_2_dem_min.tif
[3/3] Processing P4431G1_3.laz → P4431G1_3_dem_min.tif
Pipeline finished. Created 3 DEM files
